# This file gathers COUNTER data from vendors who offer usage data by directly accessing their COUNTER APIs.

## This version automatically downloads multiple reports, one after the other. It is best used for bulk usage gathering and not for troubleshooting.

In [4]:
# Load Packages

import numpy as np
import pandas as pd
import os
from os import listdir
from os.path import isfile, join
import shutil
from pprint import pprint
import csv
import sqlite3
from sqlite3 import Error
from pathlib import Path
import ast
import warnings
import xlsxwriter
import requests as req
import time
import re
import json

warnings.filterwarnings("ignore")

## SUSHI API Data by Vendor is saved in the API_Info.py file. If any APIs need updates, make changes within that file.

## Load the package in the cell below, either by connecting to the folder it is stored or putting API_Info.py in the same folder as this code.

# NOTE: Do not post that API_Info file to GitHub at any time! It must be stored in a safe, non-public location. 

In [5]:
#Use this if the API_Info file is not in the same folder as the code - otherwise, the package can just be imported
apiInfoFolderPath = r"C:\Users\aharrima\GitHub_UTK\Usage_Database"
import API_Info

# Input Key Variables for the Vendor

### Rules for Input Variables:

1. All of the variables, except for counterType, can have multiple options.
2. Even if using just one option for a variable, make sure everything except for counterType is placed between brackets []. The code will not function if a variable besides counterType is missing brackets.
3. Separate entries, when used, should be separated by commas between each item in the brackets.
4. Any vendors that require any manual steps in the middle for proper SUSHI gathering will not work in this format. Use the single-file method for those instead.
5. The more reports run, especially for one vendor or one usage platform, can increase the chance of getting overload 500 errors. If you see a 500 error appear at any time while running, cancel the code by going to Kernel -> Interrupt Kernel. Then wait for a few moments before picking up where you left off.

In [6]:
# Use this space to store any commonly-used lists for a variable in order to save on time.


#vendor = ['AAAS', 'AACCN', 'AACR', 'AAPediatrics', 'AcademyofManagement', 'AccessEngineering', 'AccessMedicine', 'ACM', 'ACS', 'AdamMatthew', 'AIAA', 'AIPScitation', 'AllenPress', 'AmerEconAssoc', 'AmerPhysicalSoc', 'AmerSocMicro', 'AmerSpeechLang', 'AmPhytoSoc',
#          'AMS', 'AnnalsIntMed', 'AnnualReviews', 'APA', 'APHA', 'APS', 'ASCB', 'ASCE', 'ASHS', 'ASME', 'ASP', 'ASTMCompass', 'Berghahn', 'BiblioScout', 'BioOne', 'Brepols', 'BrillBOPI', 'BrillOnline', 'Cabells_Journalytics', 'CABIntl', 'Cambridge',
#          'CHE_JR', 'CIAO', 'CompBiol', 'CQandSage', 'Credo', 'CSIRO', 'CSP', 'DeGruyter', 'Diabetes', 'DigitalTheatre', 'Duke', 'EastView', 'EBSCO', 'Emerald', 'Euclid', 'FirstSearch', 'Fulcrum', 'GaleUTK', 'GeoScienceWorld', 'GSP', 
#          'Guilford', 'Hanser', 'HealthAffairs', 'Highwire', 'HistoryCommons', 'Hogrefe', 'HumanKinetics', 'IEEE', 'Informs_PubsOnline', 'IngentaConnect', 'IOP', 'ISIWebofScience', 'IWA', 'JAMA', 'JBE', 'JPSJ', 'JSAD', 'JSTAGE',
#          'JSTOR', 'Karger', 'Liebert', 'Liverpool', 'LOEB', 'MicrobiologySociety', 'MIT', 'MSP', 'NEJM', 'NewsBank', 'Onepetro', 'OPG', 'OUP', 'OvidSP', 'PhilPapers', 'PLOS', 'PNAS', 'ProjectMuse', 'ProQuest', 'ProQuestEBooks',
#          'PsychOnline', 'RDAToolkit', 'Rotunda', 'RoyalSociety', 'RSC', 'RUP', 'Sabinet', 'SAE', 'SageJournals', 'SagePublications', 'SIAM', 'Slack', 'SocialExplorer', 'SPIE', 'SpringerAll', 'T&Febooks', 
#          'TelfordICE', 'TFOnline', 'Thieme', 'UCP', 'UnivChicago', 'UToronto', 'V&R', 'Wiley', 'WorldScientific']

In [7]:
vendor = ['ASP']
counterType = 5.1     #Put 5 or 5.1 here with no brackets. 5.1 began in January 2025, but some vendors converted older usage to 5.1. Others have a sharp cutoff, where pre-2025 usage is C5 and post-2024 usage is C5.1.
startDate = ['2025-07-01'] #Please use YYYY-MM-DD format for each date; while some vendors only need YYYY-MM, the full date is used later in this code.
endDate = ['2025-12-31'] 
reportType = ['DR', 'PR_P1', 'IR_M1'] #Include the underscore if you are gathering reports other than the Master Report. Example: IR_M1, PR_P1.

# The Giant Cell of Data Collection

### This cell will go through each step of the usage-gathering process. Each run will have messages showing any progress made, any reports downloaded, or any errors that caused the code to fail.

### If you need to double-check why a report, vendor, etc. did not work, it is recommended to test it individually using the single-report code.

In [12]:
# COUNTER 5 or 5.1 Check
for a in range(len(vendor)):
    print(vendor[a])
    for b in range(len(reportType)):
        print(reportType[b])
        for c in range(len(startDate)):
            print(startDate[c])
            try:
                if vendor[a] == 'ACS' or vendor[a] == 'AmerPhysicalSoc' or vendor[a] == 'APA' or vendor[a] == 'Cabells_Journalytics': #Check for vendors that only offer COUNTER 5.1
                    counterPiece = 'r51/reports/'
                elif vendor[a] == 'IngentaConnect':
                    counterPiece = ''
                elif counterType == 5.1 and vendor[a] != 'AdamMatthew' and vendor[a] != 'ASP' and vendor[a] != 'BiblioScout' and vendor[a] != 'Cambridge' and vendor[a] != 'D&H' and vendor[a] != 'Highwire' and vendor[a] != 'IOP' and vendor[a] != 'ISIWebofScience' and vendor[a] != 'MohrSiebeck' and vendor[a] != 'NewsBank' and vendor[a] != 'RSC' and vendor[a] != 'SAE':
                    counterPiece = 'r51/reports/'
                else:
                    counterPiece = 'reports/'
                
                # Requestor Check
                if API_Info.requestorIDs[vendor[a]] == '':
                    requestorPiece = ''
                else:
                    requestorPiece = r''+'requestor_id='+ API_Info.requestorIDs[vendor[a]]
                
                # Customer Check
                if API_Info.customerIDs[vendor[a]] == '':
                    customerPiece = ''
                else:
                    customerPiece = r''+'&customer_id='+API_Info.customerIDs[vendor[a]]

                # Cutomer First Parameter Check
                if vendor[a] == 'Thieme' or vendor[a] == 'SAE' or vendor[a] == 'IEEE':
                    customerPiece = r''+'customer_id='+API_Info.customerIDs[vendor[a]]
                
                # API Key Check
                if API_Info.apiKeys[vendor[a]] == '':
                    apiKeyPiece = ''
                else:
                    apiKeyPiece = r''+'&api_key='+API_Info.apiKeys[vendor[a]]
                
                # Slash Piece Correction
                if vendor[a] == 'Gale61277' or vendor[a] == 'ProjectMuse' or vendor[a] == 'Thieme' or vendor[a] == 'Wiley' or vendor[a] == 'APA' or vendor[a] == 'AMS' or vendor[a] == 'AmerPhysicalSoc' or vendor[a] == 'BiblioScout' or vendor[a] == 'BOA' or vendor[a] == 'Brepols' or vendor[a] == 'BrillOnline' or vendor[a] == 'Cabells_Journalytics' or vendor[a] == 'CHE_JR' or vendor[a] == 'CIAO' or vendor[a] == 'Credo' or vendor[a] == 'DeGruyter' or vendor[a] == 'EastView' or vendor[a] == 'FirstSearch' or vendor[a] == 'Highwire' or vendor[a] == 'HistoryCommons' or vendor[a] == 'IngentaConnect' or vendor[a] == 'JBE' or vendor[a] == 'JSTAGE' or vendor[a] == 'LOEB' or vendor[a] == 'NicrobiologySociety' or vendor[a] == 'MohrSiebeck' or vendor[a] == 'MSP' or vendor[a] == 'PLOS' or vendor[a] == 'ProjectMuse' or vendor[a] == 'ProQuestEBooks' or vendor[a] == 'RDAToolkit' or vendor[a] == 'Rotunda' or vendor[a] == 'SAE':
                    slashPiece = '?'
                else:
                    slashPiece = '/?'
                
                # Mandatory Lowercase Vendor Check
                if vendor[a] == 'AccessEngineering' or vendor[a] == 'ASTMCompass' or vendor[a] == 'BrillOnline' or vendor[a] == 'SpringerAll' or vendor[a] == 'Thieme' or vendor[a] == 'Wiley' or vendor[a] == 'ISIWebofScience' or vendor[a] == 'EBSCO' or vendor[a] == 'Credo' or vendor[a] == 'BiblioScout' or vendor[a] == 'Brepols' or vendor[a] == 'JBE' or vendor[a] == 'MicrobiologySociety' or vendor[a] == 'RSC' or vendor[a] == 'SAE' or vendor[a] == 'Cabells_Journalytics' or vendor[a] == 'IEEE' or vendor[a] == 'IOP' or vendor[a] == 'IngentaConnect':
                    reportType[b] = reportType[b].lower()
                else:
                    reportType[b] = reportType[b]

                # Mandatory YYYY-MM Vendor Check
                if vendor[a] == 'JSTAGE' or vendor[a] == 'IngentaConnect':
                    startDate[c] = startDate[c][:7]
                    endDate[c] = endDate[c][:7]
    
                requestURL = r''+API_Info.urls[vendor[a]]+counterPiece+reportType[b]+slashPiece+requestorPiece+customerPiece+apiKeyPiece+'&begin_date='+startDate[c]+'&end_date='+endDate[c]

                # Try to Gather the Data, and Gather any HTTP Errors
                try:
                    rawRequest = req.get(requestURL)
                    rawRequest
                except req.exceptions.HTTPError as http_error:
                    print(f"HTTP Error {http_error} occurred while downloading {vendor[a]}'s {reportType[b]} report.")
                    raise ValueError('An HTTP Error came from the API while gathering this report.')
                    time.sleep(5)

                # Remove any underscores from report names for naming conventions later
                if '_' in reportType[b]:
                    reportType[b] = re.sub(r'[^a-zA-Z0-9]', '', reportType[b])
        
                # Gather the Data and Convert to JSON
                data = rawRequest.json()

                if 'Exceptions' in data['Report_Header']:
                    print(f"Exceptions Found: {data['Report_Header']['Exceptions']}")
                    raise ValueError('An Exception was found gathering this report.')
        
                dataItems = data['Report_Items']
        
                print('Data Items Found')
        
                # Prepare the Columns and Clear Any Existing Information
                Platform = pd.Series([])
                DataType = pd.Series([])
                MetricType = pd.Series([])
                Month = pd.Series([])
                Usage = pd.Series([])
                Vendor = pd.Series([])
                Database = pd.Series([])
                Title = pd.Series([])
                OnlineISSN = pd.Series([])
                PrintISSN = pd.Series([])
                Publisher = pd.Series([])
                AccessType = pd.Series([])
                Item = pd.Series([])
                onlineIssnName = 'NA'
                printIssnName = 'NA'
                isbnName = 'NA'
                
                ## Checking for Various Types of JSON Downloads
                
                if 'Attribute_Performance' in dataItems or 'Attribute_Performance' in dataItems[0]: #or Attribute_Performance in dataItems[0]['Items'][0]: #This third one is specifically for IR_M1 formatting, and needs to be commented out if a different report is running.
                    functionVersion = 'Attribute_Performance'
                else:
                    functionVersion = 'Performance'
                
                if 'Data_Type' in dataItems[0]:
                    dataTypeVersion = 'Data_Type'
                else:
                    dataTypeVersion = 'Unknown'
        
                # Clean up the JSON:
        
                # Platform Report Downloads - PR or PR_P1
                
                if reportType[b] == 'PR' or reportType[b] == 'pr' or reportType[b] == 'PRP1' or reportType[b] == 'prp1':
                    if functionVersion == 'Attribute_Performance':
                        for h in range(len(dataItems)):
                            attributes = dataItems[h]['Attribute_Performance']
                            platformName = dataItems[h]['Platform']
                            for i in range(len(attributes)):
                                performanceList = attributes[i]['Performance']
                                dataType = attributes[i]['Data_Type']
                                for j in range(len(performanceList)):
                                    dictionaryItemsJ = performanceList.items()
                                    keys = list(map(lambda x: x[0], dictionaryItemsJ))
                                    metricType = keys[j]
                                    months = performanceList[keys[j]]
                                    for k in range(len(months)):
                                        dictionaryItemsK = months.items()
                                        keys2 = list(map(lambda y: y[0], dictionaryItemsK))
                                        month = keys2[k]
                                        values = months[keys2[k]]
                                        
                                        #print('Platform:', platformName)
                                        Platform = pd.concat([Platform, pd.Series([platformName])], ignore_index = True)
                                        #print('Data Type:', dataType)
                                        DataType = pd.concat([DataType, pd.Series([dataType])], ignore_index = True)
                                        #print('Metric Type:', metricType)
                                        MetricType = pd.concat([MetricType, pd.Series([metricType])], ignore_index = True)
                                        #print('Month:', month)
                                        Month = pd.concat([Month, pd.Series([month])], ignore_index = True)
                                        #print('Usage Value:', values)
                                        Usage = pd.concat([Usage, pd.Series([values])], ignore_index = True)
                        
                                        Vendor = pd.concat([Vendor, pd.Series([vendor[a]])], ignore_index = True)
                    
                    if functionVersion == 'Performance':
                        for h in range(len(dataItems)):
                            performances = dataItems[h]['Performance']
                            platformName = dataItems[h]['Platform']
                            for i in range(len(performances)):
                                performanceList = performances
                                month = performances[i]['Period']['Begin_Date']
                                dataType = performances[i]['Instance']
                                if dataTypeVersion == 'Data_Type':
                                    dataName = dataItems[h]['Data_Type']
                                else:
                                    dataName = 'Unknown'
                                for j in range(len(dataType)):
                                    metricType = dataType[j]['Metric_Type']
                                    values = dataType[j]['Count']
                                    
                                    #print('Platform:', platformName)
                                    Platform = pd.concat([Platform, pd.Series([platformName])], ignore_index = True)
                                    #print('Data Type:', dataType)
                                    DataType = pd.concat([DataType, pd.Series([dataName])], ignore_index = True)
                                    #print('Metric Type:', metricType)
                                    MetricType = pd.concat([MetricType, pd.Series([metricType])], ignore_index = True)
                                    #print('Month:', month)
                                    Month = pd.concat([Month, pd.Series([month])], ignore_index = True)
                                    #print('Usage Value:', values)
                                    Usage = pd.concat([Usage, pd.Series([values])], ignore_index = True)
                    
                                    Vendor = pd.concat([Vendor, pd.Series([vendor[a]])], ignore_index = True)    
                    
                    
                    longUsage = pd.concat([Platform, DataType, MetricType, Month, Usage, Vendor], axis = 1)
                    
                    longUsage.columns = ['Platform', 'DataType', 'MetricType', 'Month', 'Usage', 'Vendor']
                
                
                
                # Database Report Downloads - DR or DR_D1
                if reportType[b] == 'DR' or reportType[b] == 'dr' or reportType[b] == 'DRD1' or reportType[b] == 'drd1':
                    if functionVersion == 'Attribute_Performance':
                        for h in range(len(dataItems)):
                            attributes = dataItems[h]['Attribute_Performance']
                            databaseName = dataItems[h]['Database']
                            platformName = dataItems[h]['Platform']
                            for i in range(len(attributes)):
                                performanceList = attributes[i]['Performance']
                                if dataTypeVersion == 'Data_Type':
                                    dataName = dataItems[h]['Data_Type']
                                else:
                                    dataName = 'Unknown'
                                for j in range(len(performanceList)):
                                    dictionaryItemsJ = performanceList.items()
                                    keys = list(map(lambda x: x[0], dictionaryItemsJ))
                                    metricType = keys[j]
                                    months = performanceList[keys[j]]
                                    for k in range(len(months)):
                                        dictionaryItemsK = months.items()
                                        keys2 = list(map(lambda y: y[0], dictionaryItemsK))
                                        month = keys2[k]
                                        values = months[keys2[k]]
                                        
                                        #print('Platform:', platformName)
                                        Database = pd.concat([Database, pd.Series([databaseName])], ignore_index = True)
                                        Platform = pd.concat([Platform, pd.Series([platformName])], ignore_index = True)
                                        #print('Data Type:', dataType)
                                        DataType = pd.concat([DataType, pd.Series([dataType])], ignore_index = True)
                                        #print('Metric Type:', metricType)
                                        MetricType = pd.concat([MetricType, pd.Series([metricType])], ignore_index = True)
                                        #print('Month:', month)
                                        Month = pd.concat([Month, pd.Series([month])], ignore_index = True)
                                        #print('Usage Value:', values)
                                        Usage = pd.concat([Usage, pd.Series([values])], ignore_index = True)
                        
                                        Vendor = pd.concat([Vendor, pd.Series([vendor[a]])], ignore_index = True)
                    if functionVersion == 'Performance':
                        for h in range(len(dataItems)):
                            performances = dataItems[h]['Performance']
                            databaseName = dataItems[h]['Database']
                            platformName = dataItems[h]['Platform']
                            for i in range(len(performances)):
                                performanceList = performances
                                month = performances[i]['Period']['Begin_Date']
                                dataType = performances[i]['Instance']
                                if dataTypeVersion == 'Data_Type':
                                    dataName = dataItems[h]['Data_Type']
                                else:
                                    dataName = 'Unknown'
                                for j in range(len(dataType)):
                                    metricType = dataType[j]['Metric_Type']
                                    values = dataType[j]['Count']
                                    
                                    #print('Platform:', platformName)
                                    Database = pd.concat([Database, pd.Series([databaseName])], ignore_index = True)
                                    Platform = pd.concat([Platform, pd.Series([platformName])], ignore_index = True)
                                    #print('Data Type:', dataType)
                                    DataType = pd.concat([DataType, pd.Series([dataName])], ignore_index = True)
                                    #print('Metric Type:', metricType)
                                    MetricType = pd.concat([MetricType, pd.Series([metricType])], ignore_index = True)
                                    #print('Month:', month)
                                    Month = pd.concat([Month, pd.Series([month])], ignore_index = True)
                                    #print('Usage Value:', values)
                                    Usage = pd.concat([Usage, pd.Series([values])], ignore_index = True)
                    
                                    Vendor = pd.concat([Vendor, pd.Series([vendor[a]])], ignore_index = True)  

                    longUsage = pd.concat([Database, Platform, DataType, MetricType, Month, Usage, Vendor], axis = 1)
                    
                    longUsage.columns = ['Database', 'Platform', 'DataType', 'MetricType', 'Month', 'Usage', 'Vendor']
                                
                                
                # Title Master Report Downloads - TR
                if reportType[b] == 'TR' or reportType[b] == 'tr':    
                    if functionVersion == 'Attribute_Performance':
                        for h in range(len(dataItems)):
                            if vendor[a] == 'AAAS' and h == 0:
                                continue
                            onlineIssnName = 'NA'
                            printIssnName = 'NA'
                            isbnName = 'NA'
                            attributes = dataItems[h]['Attribute_Performance']
                            titleName = dataItems[h]['Title']
                            platformName = dataItems[h]['Platform']
                            if 'Publisher' in dataItems[h]:
                                publisherName = dataItems[h]['Publisher']
                            else:
                                publisherName = 'NA'
                            if 'Item_ID' in dataItems[h]:
                                itemIDs = dataItems[h]['Item_ID']
                                if 'Online_ISSN' in itemIDs:
                                    onlineIssnName = itemIDs['Online_ISSN']
                                if 'Print_ISSN' in itemIDs:
                                    printIssnName = itemIDs['Print_ISSN']
                                if 'ISBN' in itemIDs:
                                    isbnName = itemIDs['ISBN']
                            else:
                                onlineIssnName = 'NA'
                                printIssnName = 'NA'
                                isbnName = 'NA'
                            for i in range(len(attributes)):
                                performanceList = attributes[i]['Performance']
                                dataType = attributes[i]['Data_Type']
                                for j in range(len(performanceList)):
                                    dictionaryItemsJ = performanceList.items()
                                    keys = list(map(lambda x: x[0], dictionaryItemsJ))
                                    metricType = keys[j]
                                    months = performanceList[keys[j]]
                                    for k in range(len(months)):
                                        dictionaryItemsK = months.items()
                                        keys2 = list(map(lambda y: y[0], dictionaryItemsK))
                                        month = keys2[k]
                                        values = months[keys2[k]]
                                        
                                        #print('Platform:', platformName)
                                        Title = pd.concat([Title, pd.Series([titleName])], ignore_index = True)
                                        OnlineISSN = pd.concat([OnlineISSN, pd.Series([onlineIssnName])], ignore_index = True)
                                        PrintISSN = pd.concat([PrintISSN, pd.Series([printIssnName])], ignore_index = True)
                                        ISBN = pd.concat([ISBN, pd.Series([isbnName])], ignore_index = True)
                                        Publisher = pd.concat([Publisher, pd.Series([publisherName])], ignore_index = True)
                                        Platform = pd.concat([Platform, pd.Series([platformName])], ignore_index = True)
                                        #print('Data Type:', dataType)
                                        DataType = pd.concat([DataType, pd.Series([dataType])], ignore_index = True)
                                        #print('Metric Type:', metricType)
                                        MetricType = pd.concat([MetricType, pd.Series([metricType])], ignore_index = True)
                                        #print('Month:', month)
                                        Month = pd.concat([Month, pd.Series([month])], ignore_index = True)
                                        #print('Usage Value:', values)
                                        Usage = pd.concat([Usage, pd.Series([values])], ignore_index = True)
                        
                                        Vendor = pd.concat([Vendor, pd.Series([vendor[a]])], ignore_index = True)
                    
                 
                                    
                    if functionVersion == 'Performance':
                        for h in range(len(dataItems)):
                            onlineIssnName = 'NA'
                            printIssnName = 'NA'
                            isbnName = 'NA'
                            performances = dataItems[h]['Performance']
                            titleName = dataItems[h]['Title']
                            if 'Publisher' in dataItems[h]:
                                publisherName = dataItems[h]['Publisher']
                            else:
                                publisherName = 'NA'
                            if 'Item_ID' in dataItems[h]:
                                itemIDs = dataItems[h]['Item_ID']
                                for g in range(len(itemIDs)):
                                    if itemIDs[g]['Type'] == 'Online_ISSN':
                                        onlineIssnName = itemIDs[g]['Value']
                                    if itemIDs[g]['Type'] == 'Print_ISSN':
                                        printIssnName = itemIDs[g]['Value']
                                    if itemIDs[g]['Type'] == 'ISBN':
                                        isbnName = itemIDs[g]['Value']
                            else:
                                onlineIssnName = 'NA'
                                printIssnName = 'NA'
                                isbnName = 'NA'
                            if 'Platform' in dataItems[h]:
                                platformName = dataItems[h]['Platform']
                            else:
                                platformName = 'NA'
                            for i in range(len(performances)):
                                performanceList = performances
                                month = performances[i]['Period']['Begin_Date']
                                dataType = performances[i]['Instance']
                                if dataTypeVersion == 'Data_Type':
                                    dataName = dataItems[h]['Data_Type']
                                else:
                                    dataName = 'Unknown'
                                for j in range(len(dataType)):
                                    metricType = dataType[j]['Metric_Type']
                                    values = dataType[j]['Count']
                                    
                                    #print('Platform:', platformName)
                                    Title = pd.concat([Title, pd.Series([titleName])], ignore_index = True)
                                    OnlineISSN = pd.concat([OnlineISSN, pd.Series([onlineIssnName])], ignore_index = True)
                                    PrintISSN = pd.concat([PrintISSN, pd.Series([printIssnName])], ignore_index = True)
                                    ISBN = pd.concat([ISBN, pd.Series([isbnName])], ignore_index = True)
                                    Publisher = pd.concat([Publisher, pd.Series([publisherName])], ignore_index = True)
                                    Platform = pd.concat([Platform, pd.Series([platformName])], ignore_index = True)
                                    #print('Data Type:', dataType)
                                    DataType = pd.concat([DataType, pd.Series([dataName])], ignore_index = True)
                                    #print('Metric Type:', metricType)
                                    MetricType = pd.concat([MetricType, pd.Series([metricType])], ignore_index = True)
                                    #print('Month:', month)
                                    Month = pd.concat([Month, pd.Series([month])], ignore_index = True)
                                    #print('Usage Value:', values)
                                    Usage = pd.concat([Usage, pd.Series([values])], ignore_index = True)
                    
                                    Vendor = pd.concat([Vendor, pd.Series([vendor[a]])], ignore_index = True)    
                    
                    
                    longUsage = pd.concat([Title, OnlineISSN, PrintISSN, ISBN, Publisher, Platform, DataType, MetricType, Month, Usage, Vendor], axis = 1)
                    
                    longUsage.columns = ['Title', 'Online_ISSN', 'Print_ISSN', 'ISBN', 'Publisher', 'Platform', 'DataType', 'MetricType', 'Month', 'Usage', 'Vendor']
                
                    
                # Title Master Report Downloads - TR_B3 and TR_J3
                if reportType[b] == 'TRB3' or reportType[b] == 'TRJ3' or reportType[b] == 'trb3' or reportType[b] == 'trj3':        
                    if functionVersion == 'Attribute_Performance':
                        if vendor[a] == 'AAAS':
                            for h in range(1, len(dataItems)):
                                attributes = dataItems[h]['Attribute_Performance']
                                titleName = dataItems[h]['Title']
                                if 'Publisher' in dataItems[h]:
                                    publisherName = dataItems[h]['Publisher']
                                else:
                                    publisherName = 'NA'
                                platformName = dataItems[h]['Platform']
                                if 'Item_ID' in dataItems[h]:
                                    itemIDs = dataItems[h]['Item_ID']
                                    if 'Online_ISSN' in itemIDs:
                                        onlineIssnName = itemIDs['Online_ISSN']
                                    else:
                                        onlineIssnName = 'NA'
                                    if 'Print_ISSN' in itemIDs:
                                        printIssnName = itemIDs['Print_ISSN']
                                    else:
                                        onlineIssnName = 'NA'
                                else:
                                    onlineIssnName = 'NA'
                                    printIssnName = 'NA'
                                for i in range(len(attributes)):
                                    performanceList = attributes[i]['Performance']
                                    accessType = attributes[i]['Access_Type']
                                    #dataType = attributes[i]['Data_Type']
                                    for j in range(len(performanceList)):
                                        dictionaryItemsJ = performanceList.items()
                                        keys = list(map(lambda x: x[0], dictionaryItemsJ))
                                        metricType = keys[j]
                                        months = performanceList[keys[j]]
                                        for k in range(len(months)):
                                            dictionaryItemsK = months.items()
                                            keys2 = list(map(lambda y: y[0], dictionaryItemsK))
                                            month = keys2[k]
                                            values = months[keys2[k]]
                                            
                                            #print('Platform:', platformName)
                                            Title = pd.concat([Title, pd.Series([titleName])], ignore_index = True)
                                            Publisher = pd.concat([Publisher, pd.Series([publisherName])], ignore_index = True)
                                            Platform = pd.concat([Platform, pd.Series([platformName])], ignore_index = True)
                                            #print('Data Type:', dataType)
                                            #DataType = pd.concat([DataType, pd.Series([dataType])], ignore_index = True)
                                            AccessType = pd.concat([AccessType, pd.Series([accessType])], ignore_index = True)
                                            #print('Metric Type:', metricType)
                                            MetricType = pd.concat([MetricType, pd.Series([metricType])], ignore_index = True)
                                            #print('Month:', month)
                                            Month = pd.concat([Month, pd.Series([month])], ignore_index = True)
                                            #print('Usage Value:', values)
                                            Usage = pd.concat([Usage, pd.Series([values])], ignore_index = True)
                            
                                            Vendor = pd.concat([Vendor, pd.Series([vendor[a]])], ignore_index = True)
                        else:
                            for h in range(len(dataItems)):
                                attributes = dataItems[h]['Attribute_Performance']
                                titleName = dataItems[h]['Title']
                                if 'Publisher' in dataItems[h]:
                                    publisherName = dataItems[h]['Publisher']
                                else:
                                    publisherName = 'NA'
                                platformName = dataItems[h]['Platform']
                                if 'Item_ID' in dataItems[h]:
                                    itemIDs = dataItems[h]['Item_ID']
                                    if 'Online_ISSN' in itemIDs:
                                        onlineIssnName = itemIDs['Online_ISSN']
                                    else:
                                        onlineIssnName = 'NA'
                                    if 'Print_ISSN' in itemIDs:
                                        printIssnName = itemIDs['Print_ISSN']
                                    else:
                                        onlineIssnName = 'NA'
                                else:
                                    onlineIssnName = 'NA'
                                    printIssnName = 'NA'
                                for i in range(len(attributes)):
                                    performanceList = attributes[i]['Performance']
                                    accessType = attributes[i]['Access_Type']
                                    #dataType = attributes[i]['Data_Type']
                                    for j in range(len(performanceList)):
                                        dictionaryItemsJ = performanceList.items()
                                        keys = list(map(lambda x: x[0], dictionaryItemsJ))
                                        metricType = keys[j]
                                        months = performanceList[keys[j]]
                                        for k in range(len(months)):
                                            dictionaryItemsK = months.items()
                                            keys2 = list(map(lambda y: y[0], dictionaryItemsK))
                                            month = keys2[k]
                                            values = months[keys2[k]]
                                            
                                            #print('Platform:', platformName)
                                            Title = pd.concat([Title, pd.Series([titleName])], ignore_index = True)
                                            Publisher = pd.concat([Publisher, pd.Series([publisherName])], ignore_index = True)
                                            Platform = pd.concat([Platform, pd.Series([platformName])], ignore_index = True)
                                            #print('Data Type:', dataType)
                                            #DataType = pd.concat([DataType, pd.Series([dataType])], ignore_index = True)
                                            AccessType = pd.concat([AccessType, pd.Series([accessType])], ignore_index = True)
                                            #print('Metric Type:', metricType)
                                            MetricType = pd.concat([MetricType, pd.Series([metricType])], ignore_index = True)
                                            #print('Month:', month)
                                            Month = pd.concat([Month, pd.Series([month])], ignore_index = True)
                                            #print('Usage Value:', values)
                                            Usage = pd.concat([Usage, pd.Series([values])], ignore_index = True)
                            
                                            Vendor = pd.concat([Vendor, pd.Series([vendor[a]])], ignore_index = True)            
                    
                    
                    if functionVersion == 'Performance':
                        onlineIssnName = 'NA'
                        printIssnName = 'NA'
                        for h in range(len(dataItems)):
                            performances = dataItems[h]['Performance']
                            titleName = dataItems[h]['Title']
                            publisherName = dataItems[h]['Publisher']
                            if 'Item_ID' in dataItems[h]:
                                itemIDs = dataItems[h]['Item_ID']
                                for g in range(len(itemIDs)):
                                    if itemIDs[g]['Type'] == 'Online_ISSN':
                                        onlineIssnName = itemIDs[g]['Value']
                                    if itemIDs[g]['Type'] == 'Print_ISSN':
                                        printIssnName = itemIDs[g]['Value']
                            else:
                                onlineIssnName = 'NA'
                                printIssnName = 'NA'
                            accessType = dataItems[h]['Access_Type']
                            if 'Platform' in dataItems[h]:
                                platformName = dataItems[h]['Platform']
                            else:
                                platformName = 'NA'
                            for i in range(len(performances)):
                                performanceList = performances
                                month = performances[i]['Period']['Begin_Date']
                                dataType = performances[i]['Instance']
                                if dataTypeVersion == 'Data_Type':
                                    dataName = dataItems[h]['Data_Type']
                                else:
                                    dataName = 'Unknown'
                                for j in range(len(dataType)):
                                    metricType = dataType[j]['Metric_Type']
                                    values = dataType[j]['Count']
                                    
                                    #print('Platform:', platformName)
                                    Title = pd.concat([Title, pd.Series([titleName])], ignore_index = True)
                                    OnlineISSN = pd.concat([OnlineISSN, pd.Series([onlineIssnName])], ignore_index = True)
                                    PrintISSN = pd.concat([PrintISSN, pd.Series([printIssnName])], ignore_index = True)
                                    Publisher = pd.concat([Publisher, pd.Series([publisherName])], ignore_index = True)
                                    Platform = pd.concat([Platform, pd.Series([platformName])], ignore_index = True)
                                    #print('Data Type:', dataType)
                                    DataType = pd.concat([DataType, pd.Series([dataName])], ignore_index = True)
                                    AccessType = pd.concat([AccessType, pd.Series([accessType])], ignore_index = True)
                                    #print('Metric Type:', metricType)
                                    MetricType = pd.concat([MetricType, pd.Series([metricType])], ignore_index = True)
                                    #print('Month:', month)
                                    Month = pd.concat([Month, pd.Series([month])], ignore_index = True)
                                    #print('Usage Value:', values)
                                    Usage = pd.concat([Usage, pd.Series([values])], ignore_index = True)
                    
                                    Vendor = pd.concat([Vendor, pd.Series([vendor[a]])], ignore_index = True)    

                                    
                    longUsage = pd.concat([Title, OnlineISSN, PrintISSN, Publisher, Platform, AccessType, MetricType, Month, Usage, Vendor], axis = 1)
                    
                    longUsage.columns = ['Title', 'Online_ISSN', 'Print_ISSN', 'Publisher', 'Platform', 'AccessType', 'MetricType', 'Month', 'Usage', 'Vendor']
                
                
                # Multimedia Item Report Downloads - IR_M1
                if reportType[b] == 'IRM1' or reportType[b] == 'irm1':      
                    if functionVersion == 'Attribute_Performance':
                        for h in range(len(dataItems)):
                            attributes = dataItems[0]['Items'][h]['Attribute_Performance']
                            itemName = dataItems[0]['Items'][h]['Item']
                            for i in range(len(attributes)):
                                performanceList = attributes[i]['Performance']
                                dataType = attributes[i]['Data_Type']
                                for j in range(len(performanceList)):
                                    dictionaryItemsJ = performanceList.items()
                                    keys = list(map(lambda x: x[0], dictionaryItemsJ))
                                    metricType = keys[j]
                                    months = performanceList[keys[j]]
                                    for k in range(len(months)):
                                        dictionaryItemsK = months.items()
                                        keys2 = list(map(lambda y: y[0], dictionaryItemsK))
                                        month = keys2[k]
                                        values = months[keys2[k]]
                                        
                                        #print('Platform:', platformName)
                                    Item = pd.concat([Item, pd.Series([itemName])], ignore_index = True)
                                    Publisher = pd.concat([Publisher, pd.Series([publisherName])], ignore_index = True)
                                    Platform = pd.concat([Platform, pd.Series([platformName])], ignore_index = True)
                                    #print('Metric Type:', metricType)
                                    MetricType = pd.concat([MetricType, pd.Series([metricType])], ignore_index = True)
                                    #print('Month:', month)
                                    Month = pd.concat([Month, pd.Series([month])], ignore_index = True)
                                    #print('Usage Value:', values)
                                    Usage = pd.concat([Usage, pd.Series([values])], ignore_index = True)
                    
                                    Vendor = pd.concat([Vendor, pd.Series([vendor[a]])], ignore_index = True)  
                    
                    
                    if functionVersion == 'Performance':
                        for h in range(len(dataItems)):
                            performances = dataItems[h]['Performance']
                            itemName = dataItems[h]['Item']
                            platformName = dataItems[h]['Platform']
                            publisherName = dataItems[h]['Publisher']
                            for i in range(len(performances)):
                                performanceList = performances
                                month = performances[i]['Period']['Begin_Date']
                                dataType = performances[i]['Instance']
                                for j in range(len(dataType)):
                                    metricType = dataType[j]['Metric_Type']
                                    values = dataType[j]['Count']
                                    
                                    #print('Platform:', platformName)
                                    Item = pd.concat([Item, pd.Series([itemName])], ignore_index = True)
                                    Publisher = pd.concat([Publisher, pd.Series([publisherName])], ignore_index = True)
                                    Platform = pd.concat([Platform, pd.Series([platformName])], ignore_index = True)
                                    #print('Metric Type:', metricType)
                                    MetricType = pd.concat([MetricType, pd.Series([metricType])], ignore_index = True)
                                    #print('Month:', month)
                                    Month = pd.concat([Month, pd.Series([month])], ignore_index = True)
                                    #print('Usage Value:', values)
                                    Usage = pd.concat([Usage, pd.Series([values])], ignore_index = True)
                    
                                    Vendor = pd.concat([Vendor, pd.Series([vendor[a]])], ignore_index = True)  
                    
                    
                    longUsage = pd.concat([Item, Publisher, Platform, MetricType, Month, Usage, Vendor], axis = 1)
                    
                    longUsage.columns = ['Item', 'Publisher', 'Platform', 'MetricType', 'Month', 'Usage', 'Vendor']      


                # Remove All NA Columns

                longUsage = longUsage.dropna(axis=1, how='all')

                print('Long Usage Created')


                # Convert the Data to Wide Format  

                valueColumn = 'Usage'
                monthColumn = 'Month'
                indexColumns = [col for col in longUsage.columns if col != valueColumn and col != monthColumn]
                
                  
                wideUsage = longUsage.pivot_table(index = indexColumns, columns = monthColumn, values = valueColumn).reset_index().fillna(0)

                
                print('Table Pivoted')
                
                ## Fix Column Names to a Standard Format
                wideUsage = wideUsage.rename(columns = {
                                            '2021-01':'Jan-2021',
                                            '2021-02':'Feb-2021',
                                            '2021-03':'Mar-2021',
                                            '2021-04':'Apr-2021',
                                            '2021-05':'May-2021',
                                            '2021-06':'Jun-2021',
                                            '2021-07':'Jul-2021',
                                            '2021-08':'Aug-2021',
                                            '2021-09':'Sep-2021',
                                            '2021-10':'Oct-2021',
                                            '2021-11':'Nov-2021',
                                            '2021-12':'Dec-2021',                            
                                            '2022-01':'Jan-2022',
                                            '2022-02':'Feb-2022',
                                            '2022-03':'Mar-2022',
                                            '2022-04':'Apr-2022',
                                            '2022-05':'May-2022',
                                            '2022-06':'Jun-2022',
                                            '2022-07':'Jul-2022',
                                            '2022-08':'Aug-2022',
                                            '2022-09':'Sep-2022',
                                            '2022-10':'Oct-2022',
                                            '2022-11':'Nov-2022',
                                            '2022-12':'Dec-2022',
                                            '2023-01':'Jan-2023',
                                            '2023-02':'Feb-2023',
                                            '2023-03':'Mar-2023',
                                            '2023-04':'Apr-2023',
                                            '2023-05':'May-2023',
                                            '2023-06':'Jun-2023',
                                            '2023-07':'Jul-2023',
                                            '2023-08':'Aug-2023',
                                            '2023-09':'Sep-2023',
                                            '2023-10':'Oct-2023',
                                            '2023-11':'Nov-2023',
                                            '2023-12':'Dec-2023',                            
                                            '2024-01':'Jan-2024',
                                            '2024-02':'Feb-2024',
                                            '2024-03':'Mar-2024',
                                            '2024-04':'Apr-2024',
                                            '2024-05':'May-2024',
                                            '2024-06':'Jun-2024',
                                            '2024-07':'Jul-2024',
                                            '2024-08':'Aug-2024',
                                            '2024-09':'Sep-2024',
                                            '2024-10':'Oct-2024',
                                            '2024-11':'Nov-2024',
                                            '2024-12':'Dec-2024',
                                            '2025-01':'Jan-2025',
                                            '2025-02':'Feb-2025',
                                            '2025-03':'Mar-2025',
                                            '2025-04':'Apr-2025',
                                            '2025-05':'May-2025',
                                            '2025-06':'Jun-2025',
                                            '2025-07':'Jul-2025',
                                            '2025-08':'Aug-2025',
                                            '2025-09':'Sep-2025',
                                            '2025-10':'Oct-2025',
                                            '2025-11':'Nov-2025',
                                            '2025-12':'Dec-2025',                            
                                            '2026-01':'Jan-2026',
                                            '2026-02':'Feb-2026',
                                            '2026-03':'Mar-2026',
                                            '2026-04':'Apr-2026',
                                            '2026-05':'May-2026',
                                            '2026-06':'Jun-2026'})
                
                wideUsage = wideUsage.rename(columns = {
                                            '2021-01-01':'Jan-2021',
                                            '2021-02-01':'Feb-2021',
                                            '2021-03-01':'Mar-2021',
                                            '2021-04-01':'Apr-2021',
                                            '2021-05-01':'May-2021',
                                            '2021-06-01':'Jun-2021',
                                            '2021-07-01':'Jul-2021',
                                            '2021-08-01':'Aug-2021',
                                            '2021-09-01':'Sep-2021',
                                            '2021-10-01':'Oct-2021',
                                            '2021-11-01':'Nov-2021',
                                            '2021-12-01':'Dec-2021',
                                            '2022-01-01':'Jan-2022',
                                            '2022-02-01':'Feb-2022',
                                            '2022-03-01':'Mar-2022',
                                            '2022-04-01':'Apr-2022',
                                            '2022-05-01':'May-2022',
                                            '2022-06-01':'Jun-2022',
                                            '2022-07-01':'Jul-2022',
                                            '2022-08-01':'Aug-2022',
                                            '2022-09-01':'Sep-2022',
                                            '2022-10-01':'Oct-2022',
                                            '2022-11-01':'Nov-2022',
                                            '2022-12-01':'Dec-2022',
                                            '2023-01-01':'Jan-2023',
                                            '2023-02-01':'Feb-2023',
                                            '2023-03-01':'Mar-2023',
                                            '2023-04-01':'Apr-2023',
                                            '2023-05-01':'May-2023',
                                            '2023-06-01':'Jun-2023',
                                            '2023-07-01':'Jul-2023',
                                            '2023-08-01':'Aug-2023',
                                            '2023-09-01':'Sep-2023',
                                            '2023-10-01':'Oct-2023',
                                            '2023-11-01':'Nov-2023',
                                            '2023-12-01':'Dec-2023',
                                            '2024-01-01':'Jan-2024',
                                            '2024-02-01':'Feb-2024',
                                            '2024-03-01':'Mar-2024',
                                            '2024-04-01':'Apr-2024',
                                            '2024-05-01':'May-2024',
                                            '2024-06-01':'Jun-2024',
                                            '2024-07-01':'Jul-2024',
                                            '2024-08-01':'Aug-2024',
                                            '2024-09-01':'Sep-2024',
                                            '2024-10-01':'Oct-2024',
                                            '2024-11-01':'Nov-2024',
                                            '2024-12-01':'Dec-2024',
                                            '2025-01-01':'Jan-2025',
                                            '2025-02-01':'Feb-2025',
                                            '2025-03-01':'Mar-2025',
                                            '2025-04-01':'Apr-2025',
                                            '2025-05-01':'May-2025',
                                            '2025-06-01':'Jun-2025',
                                            '2025-07-01':'Jul-2025',
                                            '2025-08-01':'Aug-2025',
                                            '2025-09-01':'Sep-2025',
                                            '2025-10-01':'Oct-2025',
                                            '2025-11-01':'Nov-2025',
                                            '2025-12-01':'Dec-2025',
                                            '2026-01-01':'Jan-2026',
                                            '2026-02-01':'Feb-2026',
                                            '2026-03-01':'Mar-2026',
                                            '2026-04-01':'Apr-2026',
                                            '2026-05-01':'May-2026',
                                            '2026-06-01':'Jun-2026'})

                print('Columns Renamed')
                
                # Add the Reporting_Period_Total to the final table
                if reportType[a] == 'DR' or reportType[a] == 'dr':
                    wideUsage['Reporting_Period_Total'] = wideUsage.iloc[:, 5:].sum(axis = 1)
                elif reportType[a] == 'PR' or reportType[a] == 'pr':
                    wideUsage['Reporting_Period_Total'] = wideUsage.iloc[:, 4:].sum(axis = 1)
                elif reportType[a] == 'TR' or reportType[a] == 'tr':
                    wideUsage['Reporting_Period_Total'] = wideUsage.iloc[:, 9:].sum(axis = 1)
                elif reportType[a] == 'TRB3' or reportType[a] == 'trb3':
                    wideUsage['Reporting_Period_Total'] = wideUsage.iloc[:, 9:].sum(axis = 1)
                elif reportType[a] == 'TRJ3' or reportType[a] == 'trj3':
                    wideUsage['Reporting_Period_Total'] = wideUsage.iloc[:, 9:].sum(axis = 1)
                elif reportType[a] == 'IRM1' or reportType[a] == 'irm1':
                    wideUsage['Reporting_Period_Total'] = wideUsage.iloc[:, 5:].sum(axis = 1)
                elif reportType[a] == 'DRD1' or reportType[a] == 'drd1':
                    wideUsage['Reporting_Period_Total'] = wideUsage.iloc[:, 5:].sum(axis = 1)
                elif reportType[a] == 'PRP1' or reportType[a] == 'prp1':
                    wideUsage['Reporting_Period_Total'] = wideUsage.iloc[:, 4:].sum(axis = 1)


                print('Reporting Period Total Added')
        
                ## Create the Date Piece for Export Name Based on the Initial Date
                
                renameStartDateDict = {'2021-01-01':'Jan2021',
                                            '2021-07-01':'Jul2021',
                                            '2022-01-01':'Jan2022',
                                            '2022-07-01':'Jul2022',
                                            '2023-01-01':'Jan2023',
                                            '2023-02-01':'Feb2023',
                                            '2023-03-01':'Mar2023',
                                            '2023-04-01':'Apr2023',
                                            '2023-05-01':'May2023',
                                            '2023-06-01':'Jun2023',
                                            '2023-07-01':'Jul2023',
                                            '2023-08-01':'Aug2023',
                                            '2023-09-01':'Sep2023',
                                            '2023-10-01':'Oct2023',
                                            '2023-11-01':'Nov2023',
                                            '2023-12-01':'Dec2023',
                                            '2024-01-01':'Jan2024',
                                            '2024-02-01':'Feb2024',
                                            '2024-03-01':'Mar2024',
                                            '2024-04-01':'Apr2024',
                                            '2024-05-01':'May2024',
                                            '2024-06-01':'Jun2024',
                                            '2024-07-01':'Jul2024',
                                            '2024-08-01':'Aug2024',
                                            '2024-09-01':'Sep2024',
                                            '2024-10-01':'Oct2024',
                                            '2024-11-01':'Nov2024',
                                            '2024-12-01':'Dec2024',
                                            '2025-01-01':'Jan2025',
                                            '2025-02-01':'Feb2025',
                                            '2025-03-01':'Mar2025',
                                            '2025-04-01':'Apr2025',
                                            '2025-05-01':'May2025',
                                            '2025-06-01':'Jun2025',
                                            '2025-07-01':'Jul2025',
                                            '2025-08-01':'Aug2025',
                                            '2025-09-01':'Sep2025',
                                            '2025-10-01':'Oct2025',
                                            '2025-11-01':'Nov2025',
                                            '2025-12-01':'Dec2025',    
                                            '2026-01-01':'Jan2026',
                                            '2026-02-01':'Feb2026',}
                
                renameEndDateDict = {'2021-06-30':'Jun2021',
                                            '2021-12-31':'Dec2021',
                                            '2022-06-30':'Jun2022',
                                            '2022-12-31':'Dec2022',
                                            '2023-01-31':'Jan2023',
                                            '2023-02-28':'Feb2023',
                                            '2023-03-31':'Mar2023',
                                            '2023-04-30':'Apr2023',
                                            '2023-05-31':'May2023',
                                            '2023-06-30':'Jun2023',
                                            '2023-10-31':'Oct2023',
                                            '2023-11-30':'Nov2023',
                                            '2023-12-31':'Dec2023',
                                            '2024-01-31':'Jan2024',
                                            '2024-02-29':'Feb2024',
                                            '2024-03-31':'Mar2024',
                                            '2024-04-30':'Apr2024',
                                            '2024-05-31':'May2024',
                                            '2024-06-30':'Jun2024',
                                            '2024-07-31':'Jul2024',
                                            '2024-08-31':'Aug2024',
                                            '2024-09-30':'Sep2024',
                                            '2024-10-31':'Oct2024',
                                            '2024-11-30':'Nov2024',
                                            '2024-12-31':'Dec2024',
                                            '2025-01-31':'Jan2025',
                                            '2025-02-28':'Feb2025',
                                            '2025-03-31':'Mar2025',
                                            '2025-04-30':'Apr2025',
                                            '2025-05-31':'May2025',
                                            '2025-06-30':'Jun2025',
                                            '2025-10-31':'Oct2025',
                                            '2025-11-30':'Nov2025',
                                            '2025-12-31':'Dec2025',
                                            '2026-01-31':'Jan2026',
                                            '2026-03-31':'Mar2026',
                                            '2026-05-31':'May2026',
                                            '2026-06-30':'Jun2026'}
                
        
        
                # Set the Export Folder Path
                folderPath = r"C:\Users\aharrima\GitHub_UTK\Usage_Database" # Change this path for your computer as needed. If you don't need to save locally, comment this out.
                
                directory = os.chdir(folderPath)
        
                
                
                # Wide File (One Column per Month)
                wideUsage.to_excel(r''+vendor[a]+'-'+reportType[b]+'-'+renameStartDateDict[startDate[c]]+'-'+renameEndDateDict[endDate[c]]+'.xlsx', index = False)
                
                # Long File (One Row per Month)
                #longUsage.to_excel(r''+vendor+'-'+reportType+'-'+renameStartDateDict[startDate]+'-'+renameEndDateDict[endDate]+'-Long.xlsx', index = False)
    
                print(r''+vendor[a]+'-'+reportType[b]+'-'+renameStartDateDict[startDate[c]]+'-'+renameEndDateDict[endDate[c]]+'.xlsx Saved to the Export Folder.')
                time.sleep(10)
    
            except Exception as other_error:
                print(f"Other Error {other_error} occurred while downloading {vendor[a]}'s {reportType[b]} report for period {startDate[c]} - {endDate[c]}.")
                raise ValueError('An Error was found gathering this report.')
                time.sleep(5)

print('All Requested Usage Downloaded.')

ASP
DR
2025-07-01
DR
Data Items Found
Long Usage Created
Table Pivoted
Columns Renamed
Reporting Period Total Added
ASP-DR-Jul2025-Dec2025.xlsx Saved to the Export Folder.
PR_P1
2025-07-01
PRP1
Data Items Found
Long Usage Created
Table Pivoted
Columns Renamed
Reporting Period Total Added
ASP-PRP1-Jul2025-Dec2025.xlsx Saved to the Export Folder.
IR_M1
2025-07-01
IRM1
Data Items Found
Long Usage Created
Table Pivoted
Columns Renamed
Reporting Period Total Added
ASP-IRM1-Jul2025-Dec2025.xlsx Saved to the Export Folder.
All Requested Usage Downloaded.


## Next Steps:

### Gathering More Usage in Bulk?

Then move on to the next reports. Don't forget to double-check any reports that did not properly download before moving on to your next usage gathering!


### Updating the COUNTER 5/5.1 Database?

Take this usage and apply them to the "Uploading COUNTER Data to SQLite3" file. Make sure all of these files are in a folder named "From Python" to ensure proper loading into "Uploading COUNTER Data."


### Ready to upload usage?

Place it in its proper usage folder(s) in SharePoint. If this is part of a semi-annual bulk usage gathering, make sure to mark off these vendors as gathered in Usage Master Vendor Report.